# LDA and QDA

This notebook is intentionally self-contained. It does not import local utility files, so the statistical idea, simulation, Python functions, evaluation, plots, and exam translation are all visible in one place.

## What problem are we solving?

LDA and QDA classify observations by estimating class-conditional Gaussian distributions.

**Highest-probability exam pattern:** Compare LDA, QDA, and a baseline classifier by cross-validated accuracy, then connect the result to covariance assumptions.

## Assumptions, intuition, and theory

LDA assumes classes share a covariance matrix and therefore has linear boundaries. QDA allows each class its own covariance matrix and therefore can create curved boundaries, but it estimates more parameters.

## Python method notes and documentation

`LinearDiscriminantAnalysis.fit` estimates class means and a shared covariance structure, `QuadraticDiscriminantAnalysis.fit` estimates class-specific covariance structures, and stratified CV keeps class proportions stable.

Documentation links:

- [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [LinearDiscriminantAnalysis](https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.LinearDiscriminantAnalysis.html)
- [QuadraticDiscriminantAnalysis](https://scikit-learn.org/stable/modules/generated/sklearn.discriminant_analysis.QuadraticDiscriminantAnalysis.html)
- [StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- [make_classification](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html)
- [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html)
- [make_blobs](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html)

## Fully self-contained worked example

Every code line below is commented so it is easy to scan under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for method comparison.
import matplotlib.pyplot as plt  # Import plotting tools.
from sklearn.base import clone  # Import clone for fresh estimator fits.
from sklearn.datasets import make_classification, make_moons  # Import classification simulators.
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis  # Import LDA and QDA classifiers.
from sklearn.linear_model import LogisticRegression  # Import logistic regression as a baseline.
from sklearn.metrics import accuracy_score  # Import classification accuracy.
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split  # Import stratified CV and splitting tools.
RANDOM_SEED = 123  # Store the reproducibility seed.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_gaussian_classification_data(n=180, class_sep=1.3, random_state=123):  # Define a two-class Gaussian-like simulator.
    X, y = make_classification(  # Generate a labeled binary classification data set.
        n_samples=n,  # Set the number of observations.
        n_features=2,  # Keep two predictors so boundaries can be plotted.
        n_informative=2,  # Make both features informative.
        n_redundant=0,  # Avoid redundant features.
        n_clusters_per_class=1,  # Use one cluster per class for clean geometry.
        class_sep=class_sep,  # Control class separation.
        random_state=random_state,  # Fix the simulation seed.
    )  # Finish the simulator call.
    return X, y  # Return predictors and labels.

def make_moon_classification_data(n=200, noise=0.25, random_state=123):  # Define a nonlinear classification simulator.
    X, y = make_moons(  # Generate interlocking moon-shaped classes.
        n_samples=n,  # Set the number of observations.
        noise=noise,  # Add noise to make the problem realistic.
        random_state=random_state,  # Fix the simulation seed.
    )  # Finish the make_moons call.
    return X, y  # Return predictors and labels.

def train_test_accuracy(model, X, y, test_size=0.30, random_state=123):  # Define a local classification train/test evaluator.
    X_train, X_test, y_train, y_test = train_test_split(  # Split before fitting to protect the test set.
        X,  # Pass predictor matrix.
        y,  # Pass class labels.
        test_size=test_size,  # Hold out this fraction for testing.
        random_state=random_state,  # Make the split reproducible.
        stratify=y,  # Preserve class proportions across train and test sets.
    )  # Finish the split call.
    fitted_model = clone(model)  # Clone the model so repeated calls start fresh.
    fitted_model.fit(X_train, y_train)  # Fit using training observations only.
    train_pred = fitted_model.predict(X_train)  # Predict training labels.
    test_pred = fitted_model.predict(X_test)  # Predict test labels.
    train_accuracy = accuracy_score(y_train, train_pred)  # Compute training accuracy.
    test_accuracy = accuracy_score(y_test, test_pred)  # Compute held-out accuracy.
    return fitted_model, train_accuracy, test_accuracy, y_test, test_pred  # Return model, metrics, and test predictions.


In [ ]:
X, y = make_gaussian_classification_data(n=200, class_sep=1.0, random_state=RANDOM_SEED)  # Simulate data close to LDA assumptions.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)  # Define stratified 5-fold CV.
models = {  # Create a method dictionary for comparison.
    'LDA': LinearDiscriminantAnalysis(),  # Add LDA with shared covariance assumption.
    'QDA': QuadraticDiscriminantAnalysis(),  # Add QDA with class-specific covariance assumption.
    'Logistic': LogisticRegression(max_iter=1000),  # Add logistic regression as a discriminative baseline.
}  # End the model dictionary.
rows = []  # Create a list for CV accuracy rows.
for name, model in models.items():  # Loop through the candidate classifiers.
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')  # Estimate accuracy by stratified CV.
    rows.append({'method': name, 'mean_accuracy': np.mean(scores), 'std_accuracy': np.std(scores, ddof=1)})  # Store mean and variability.
results = pd.DataFrame(rows)  # Convert comparison rows to a DataFrame.
display(results)  # Display the method-comparison table.
best_name = results.loc[results['mean_accuracy'].idxmax(), 'method']  # Select the method with highest CV accuracy.
print(f'Best method by CV accuracy: {best_name}')  # Print the selected classifier.


## Exam-style problem prompt

A prompt describes Gaussian classes and asks whether LDA or QDA is preferable. Compare CV accuracy and explain the shared-versus-different covariance tradeoff.

## Adaptable code pattern

```python
# Step 1: Check whether the response is categorical.
# Step 2: Fit LDA and QDA candidates.
# Step 3: Estimate accuracy by stratified CV.
# Step 4: Choose the method with higher CV accuracy.
# Step 5: Interpret LDA as lower variance and QDA as more flexible.
```

## What to remember for the exam

LDA is simpler and often better with small samples. QDA is more flexible and can win when class covariance patterns differ enough.